# Integration Test — 3-Agent Medical Pipeline

**Runtime required:** GPU (T4 or better)  
**Before running:** Upload adapter folders to Google Drive (see Cell 2)

Pipeline: Patient text → Symptom Classifier → Appointment Retriever (DynamoDB) → Response Generator

## Cell 1 — Install dependencies

In [2]:
!pip install -q transformers peft bitsandbytes accelerate boto3 pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.0 MB/s eta 0:00:00


## Cell 2 — Mount Google Drive

Upload these two folders to your Google Drive **before** running this cell:
- `symptom_classifier_adapter/final_adapter/`  →  put at `My Drive/medi-agent/symptom_classifier_adapter/final_adapter/`
- `response_generator_adapter/final_adapter/`  →  put at `My Drive/medi-agent/response_generator_adapter/final_adapter/`

In [3]:
from google.colab import drive
drive.mount('/content/drive')

CLASSIFIER_ADAPTER = '/content/drive/MyDrive/symptom_classifier_adapter/final_adapter'
GENERATOR_ADAPTER  = '/content/drive/MyDrive/response_generator_adapter/final_adapter'

import os
assert os.path.isdir(CLASSIFIER_ADAPTER), f'Not found: {CLASSIFIER_ADAPTER}'
assert os.path.isdir(GENERATOR_ADAPTER),  f'Not found: {GENERATOR_ADAPTER}'
print('Adapter paths OK')

Mounted at /content/drive
Adapter paths OK


## Cell 3 — AWS credentials

Set these secrets in Colab → **Secrets** (🔑 icon on the left sidebar):
- `AWS_ACCESS_KEY_ID`
- `AWS_SECRET_ACCESS_KEY`
- `AWS_REGION`  (e.g. `us-west-2`)
- `DYNAMODB_TABLE_NAME`  (e.g. `DoctorSchedule`)

In [4]:
from google.colab import userdata
import os

os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_REGION']            = 'us-west-2'
os.environ['DYNAMODB_TABLE_NAME']   = 'DoctorSchedule'
print('AWS credentials loaded')

AWS credentials loaded


## Cell 4 — Define schemas and utilities (inline, no file upload needed)

In [5]:
import re
from typing import Literal
from pydantic import BaseModel

BASE_MODEL = 'meta-llama/Llama-3.2-3B-Instruct'

# ── Schemas ────────────────────────────────────────────────────────────────
class SymptomInput(BaseModel):
    patient_text: str

class ClassifierOutput(BaseModel):
    department: str
    urgency: Literal['Routine', 'Urgent', 'Emergency']

class AppointmentQuery(BaseModel):
    department: str

class AppointmentOutput(BaseModel):
    doctor: str
    time_slot: str

class ResponseInput(BaseModel):
    patient_text: str
    department: str
    doctor: str
    time_slot: str
    urgency: Literal['Routine', 'Urgent', 'Emergency']

class ResponseOutput(BaseModel):
    confirmation: str
    instructions: str

# ── Parsing ────────────────────────────────────────────────────────────────
VALID_DEPARTMENTS = {
    'Cardiology', 'Neurology', 'Dermatology', 'Gastroenterology',
    'Endocrinology', 'Pulmonology', 'Infectious Disease',
    'Orthopedics', 'Urology', 'General Medicine',
}
VALID_URGENCIES = {'Routine', 'Urgent', 'Emergency'}

def parse_classifier_output(text: str) -> ClassifierOutput:
    dept_match = re.search(r'Department:\s*([^\n]+)', text)
    urg_match  = re.search(r'Urgency:\s*([^\n]+)', text)
    department = dept_match.group(1).strip() if dept_match else 'General Medicine'
    urgency    = urg_match.group(1).strip()  if urg_match  else 'Routine'
    if department not in VALID_DEPARTMENTS: department = 'General Medicine'
    if urgency    not in VALID_URGENCIES:   urgency    = 'Routine'
    return ClassifierOutput(department=department, urgency=urgency)

def parse_response_output(text: str) -> ResponseOutput:
    conf_match = re.search(r'Confirmation:\s*(.+?)(?:\n|$)', text)
    inst_match = re.search(r'Instructions:\s*(.+)', text, re.DOTALL)
    confirmation = conf_match.group(1).strip() if conf_match else text
    instructions = inst_match.group(1).strip() if inst_match else ''
    return ResponseOutput(confirmation=confirmation, instructions=instructions)

print('Schemas and parsing utilities ready')

Schemas and parsing utilities ready


## Cell 5 — Appointment Retriever (DynamoDB)

In [6]:
import boto3
from boto3.dynamodb.conditions import Key

TABLE_NAME = os.environ.get('DYNAMODB_TABLE_NAME', 'DoctorSchedule')
AWS_REGION = os.environ.get('AWS_REGION', 'us-west-2')
DAY_ORDER  = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4}

class AppointmentRetrieverAgent:
    def __init__(self):
        self.dynamodb = boto3.resource('dynamodb', region_name=AWS_REGION)
        self.table = self.dynamodb.Table(TABLE_NAME)

    def retrieve(self, query: AppointmentQuery) -> AppointmentOutput:
        response = self.table.query(
            KeyConditionExpression=Key('department').eq(query.department),
            FilterExpression='available = :avail',
            ExpressionAttributeValues={':avail': True},
        )
        items = response.get('Items', [])
        if not items:
            return AppointmentOutput(doctor='No available doctor', time_slot='No available slot')
        items.sort(key=lambda x: (DAY_ORDER.get(x['day'], 99), x['time_slot']))
        chosen = items[0]
        return AppointmentOutput(
            doctor=chosen['doctor'],
            time_slot=f"{chosen['day']} at {chosen['time_slot']}",
        )

retriever = AppointmentRetrieverAgent()
print('AppointmentRetrieverAgent ready')

AppointmentRetrieverAgent ready


## Cell 6 — Load models (4-bit quantization for T4 GPU)

In [ ]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login
from google.colab import userdata

# ── HuggingFace authentication ─────────────────────────────────────────────
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print('HuggingFace login OK')

# ── 4-bit quantization config ──────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

def load_model(adapter_path: str):
    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL,
        token=HF_TOKEN,
    )
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map='auto',
        token=HF_TOKEN,
    )
    model = PeftModel.from_pretrained(base, adapter_path)
    model.eval()
    return tokenizer, model

print('[1/2] Loading Symptom Classifier...')
t0 = time.time()
cls_tokenizer, cls_model = load_model(CLASSIFIER_ADAPTER)
print(f'  Loaded in {time.time()-t0:.1f}s')

print('[2/2] Loading Response Generator...')
t0 = time.time()
gen_tokenizer, gen_model = load_model(GENERATOR_ADAPTER)
print(f'  Loaded in {time.time()-t0:.1f}s')

## Cell 7 — Inference helpers

In [8]:
@torch.inference_mode()
def classify(patient_text: str) -> ClassifierOutput:
    system_prompt = (
        'You are a medical triage assistant. Classify the patient\'s symptoms '
        'into a medical department and urgency level. Respond in exactly this format:\n'
        'Department: <department>\n'
        'Urgency: <Routine|Urgent|Emergency>'
    )
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': patient_text},
    ]
    tokenized = cls_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True
    ).to(cls_model.device)
    input_len = tokenized['input_ids'].shape[-1]
    outputs = cls_model.generate(
        **tokenized, max_new_tokens=50, temperature=0.1,
        do_sample=True, pad_token_id=cls_tokenizer.eos_token_id,
    )
    generated = cls_tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return parse_classifier_output(generated)


@torch.inference_mode()
def generate_response(resp_input: ResponseInput) -> ResponseOutput:
    system_prompt = (
        'You are a compassionate medical assistant. A patient has been assigned '
        'an appointment. Write a warm, clear appointment confirmation and practical '
        'pre-visit instructions. Keep the tone professional but reassuring. '
        'Format your response as:\n'
        'Confirmation: <one sentence confirming the appointment>\n'
        'Instructions: <2-4 specific pre-visit instructions>'
    )
    user_content = (
        f'Patient symptoms: {resp_input.patient_text}\n'
        f'Assigned department: {resp_input.department}\n'
        f'Doctor: {resp_input.doctor}\n'
        f'Appointment: {resp_input.time_slot}\n'
        f'Urgency: {resp_input.urgency}'
    )
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': user_content},
    ]
    tokenized = gen_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True
    ).to(gen_model.device)
    input_len = tokenized['input_ids'].shape[-1]
    outputs = gen_model.generate(
        **tokenized, max_new_tokens=512, temperature=0.7,
        do_sample=True, top_p=0.9, pad_token_id=gen_tokenizer.eos_token_id,
    )
    generated = gen_tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return parse_response_output(generated)

print('Inference helpers ready')

Inference helpers ready


## Cell 8 — Run integration tests

In [9]:
TEST_CASES = [
    {
        'input': 'I have been experiencing severe chest pain and shortness of breath for the past two hours.',
        'expected_dept': 'Cardiology',
        'expected_urgency': 'Emergency',
    },
    {
        'input': 'I have a mild headache and occasional dizziness for a week.',
        'expected_dept': 'Neurology',
        'expected_urgency': 'Routine',
    },
    {
        'input': 'I have a rash on my arms that has been spreading and itching for three days.',
        'expected_dept': 'Dermatology',
        'expected_urgency': 'Routine',
    },
    {
        'input': "I've been having sharp stomach pain after eating, with nausea and bloating.",
        'expected_dept': 'Gastroenterology',
        'expected_urgency': 'Urgent',
    },
    {
        'input': 'I have a persistent cough with blood and difficulty breathing.',
        'expected_dept': 'Pulmonology',
        'expected_urgency': 'Emergency',
    },
]

print('=' * 60)
print('INTEGRATION TEST (4-bit + CUDA)')
print('=' * 60)

passed = 0
for i, case in enumerate(TEST_CASES, 1):
    print(f'\n--- Test {i}/{len(TEST_CASES)} ---')
    print(f'Input: {case["input"]}')
    t0 = time.time()

    # Agent 1: Classify
    classification = classify(case['input'])
    dept    = classification.department
    urgency = classification.urgency

    # Agent 2: Retrieve appointment
    appointment = retriever.retrieve(AppointmentQuery(department=dept))

    # Agent 3: Generate response
    resp_input = ResponseInput(
        patient_text=case['input'],
        department=dept,
        doctor=appointment.doctor,
        time_slot=appointment.time_slot,
        urgency=urgency,
    )
    response = generate_response(resp_input)
    elapsed = time.time() - t0

    # Validate
    errors = []
    if dept    not in VALID_DEPARTMENTS: errors.append(f'Invalid department: {dept}')
    if urgency not in VALID_URGENCIES:   errors.append(f'Invalid urgency: {urgency}')
    if not appointment.doctor:           errors.append('Doctor is empty')
    if not response.confirmation:        errors.append('Confirmation is empty')
    if not response.instructions:        errors.append('Instructions is empty')

    if not errors:
        passed += 1

    dept_match = dept    == case['expected_dept']
    urg_match  = urgency == case['expected_urgency']

    print(f"  Department:    {dept} {'[ok]' if dept_match else '[expected: ' + case['expected_dept'] + ']'}")
    print(f"  Urgency:       {urgency} {'[ok]' if urg_match else '[expected: ' + case['expected_urgency'] + ']'}")
    print(f'  Doctor:        {appointment.doctor}')
    print(f'  Time Slot:     {appointment.time_slot}')
    print(f'  Confirmation:  {response.confirmation[:120]}')
    print(f'  Instructions:  {response.instructions[:120]}')
    print(f'  Time:          {elapsed:.1f}s')
    print(f"  Status:        {'PASS' if not errors else 'FAIL'}")
    for e in errors:
        print(f'    ERROR: {e}')

print('\n' + '=' * 60)
print(f'RESULTS: {passed}/{len(TEST_CASES)} passed')
print('=' * 60)

INTEGRATION TEST (4-bit + CUDA)

--- Test 1/5 ---
Input: I have been experiencing severe chest pain and shortness of breath for the past two hours.
  Department:    Cardiology [ok]
  Urgency:       Urgent [expected: Emergency]
  Doctor:        Dr. Chen Wei
  Time Slot:     Monday at 08:00
  Confirmation:  Your appointment with Dr. Chen Wei in Cardiology has been confirmed for Monday at 08:00. We're here to help with your co
  Instructions:  Thanks for your question on HCM.In my opinion you should first consult cardiologist and get done 2d echo and stress test
  Time:          87.1s
  Status:        PASS

--- Test 2/5 ---
Input: I have a mild headache and occasional dizziness for a week.
  Department:    Cardiology [expected: Neurology]
  Urgency:       Routine [ok]
  Doctor:        Dr. Chen Wei
  Time Slot:     Monday at 08:00
  Confirmation:  Your appointment with Dr. Chen Wei in Cardiology has been confirmed for Monday at 08:00. We're here to help with your co
  Instructions:  Hello,